# Transformer API

In [1]:
import torch
import torch.nn as nn

## 自注意力机制模拟

In [4]:
seq_len = 10
dk =64 
Q = torch.randn([seq_len,dk])
K =  torch.randn([seq_len,dk])
V = torch.randn([seq_len,dk])
Q.shape,K.shape,V.shape

(torch.Size([10, 64]), torch.Size([10, 64]), torch.Size([10, 64]))

In [5]:
scores = Q @ K.T / torch.sqrt(torch.tensor(dk))
scores.shape # (seq_len,word_vector)

torch.Size([10, 10])

In [5]:
weights = torch.softmax(scores,dim=-1)
weights.shape

torch.Size([10, 10])

In [6]:
Attention = weights @ V
Attention.shape

torch.Size([10, 64])

## Transformer API

nn.Transformer() 可以创建一个完整的Transformer模型，包括编码器、解码器、注意力机制、位置编码等。参数如下:
- d_model: 词嵌入维度，也是词向量维度，也是模型的输出维度。
- nhead: 多头注意力机制的头数。默认为8
- num_encoder_layers:编码器的层数。默认为6
- num_decoder_layers:解码器的层数。默认为6
- dim_feedforward:前馈神经网络的中间层维度。默认为2048
- dropout: Dropout的概率。默认为0.1，用在每一个子层的输出后面，都会有dropout。
- activation:前馈神经网络中的激活函数。默认为ReLU()
- custom_encoder:自定义编码器。如果不指定，则使用TransformerEncoder。
- custom_decoder:自定义解码器。如果不指定，则使用TransformerDecoder。
- batch_first:如果为True，则输入和输出的Tensor的第一个维度是batch_size，否则是seq_len。默认为False。(推荐为True)
- layer_norm_eps:层归一化组件中的eps值(默认为1e-5)
- norm_first:如果为True，编码器和解码器将在其他注意力和前馈操作之前执行层归一化(LayerNorm)，否则在后执行，默认为False。
- bias：如果为False，则Linear层和LayerNorm中不会有偏置，默认为True。

In [8]:
transformer = nn.Transformer(d_model = 512, 
                         nhead=8,
                         num_decoder_layers=6,
                         num_encoder_layers=6,
                         dim_feedforward=2048,
                         dropout=0.1,
                         activation='relu',
                         batch_first=True,
                         norm_first=False,
                         bias=True)

nn.Transformer封装了完整的向前传播逻辑，其forward()方法定义了编码器->解码器的执行流程。该函数接收源语言序列(src_sequence，编码器输入)和目标语言序列(tgt_sequence，解码器输入)作为输入，以解码器预测结果作为输出。<br>
<img src="../resource/images/forward逻辑.png"><br>
所以nn.Transformer的forward()通常只用于训练，在推理阶段通常不会用forward()方法

nn.Transformer.forward()参数如下:
- src:encoder输入序列  (batch_size,src_seq_len,d_model)
- tgt:decoder输入序列  (batch_size,tgt_seq_len,d_model)
-  src_key_padding_mask:encoder输入序列的padding mask  (batch_size,src_seq_len),用于编码器中的自注意力机制，用以屏蔽源序列中填充(\<pad>)的位置，防止模型在计算注意力时关注无意义的内容。
-  tgt_mask:用于decoder的masked注意力子层，常用于训练阶段的自回归任务，防止模型关注当前位置之后的token，使得模型更有可能预测正确的下一个token。是一个形状为(batch_size,tgt_seq_len,tgt_seq_len) ,每一批都为一个上三角矩阵，类型为float，遮掩位置为-inf，其余为0。也支持bool类型(True表示遮挡),内部自动转为加性掩码。
-  memory_key_padding_mask: 用于解码器的交叉注意力子层，屏蔽编码器输出中的\<pad>位置，防止解码器关注源序列中的无效token。形状为()

src 与 tgt如下图所示:<br>
<img  src="../resource/images/src_seq与tgt_seq.png"  style="width: 600px;height:300px;">

src_key_padding_mask 如下图所示:<br>
<img  src="../resource/images/src_key_padding_mask.png"  style="width: 800px;height:300px;">

tgt_mask如下图所示：<br>
<img  src="../resource/images/tgt_mask.png"  style="width: 800px;height:300px;">

memory_key_padding_mask如下图所示:
<img  src="../resource/images/memory_key_padding_mask.png"  style="width: 800px;height:300px;">

nn.Transformer.forward()会输出一个(batch_size,tgt_seq_len,d_model)的张量。通常这个张量会送入线性层和softmax层，用于生成词表上的预测概率。